# GeoLife CP2 — Home / Office / POI Baseline

**Mục tiêu:** xây một baseline Home / Office có thể giải thích được, bắt đầu từ frozen CP1 stay events chứ không từ raw GPS.

Notebook này trả lời lần lượt các câu hỏi:

1. user có đủ lịch sử để suy semantic location chưa;
2. các stays có lặp lại thành recurring locations hay không;
3. mỗi stay phải được đọc theo **giờ địa phương nào**;
4. DBSCAN và complete-link khác nhau ra sao khi gom recurring location;
5. HOME/OFFICE evidence nên được tính như thế nào;
6. khi nào model nên **abstain** thay vì cố gán nhãn;
7. notebook mới khác production CP2 v1 hiện tại ở đâu.

### Pipeline cuối của notebook

```text
CP1 cleaning / stay detection
        ↓
5,821 stay events
        ↓
history sufficiency
        ↓
(lat, lon) của từng stay
        ↓
IANA timezone lookup
        ↓
local time của chính stay đó
        ↓
recurring-location clustering
        ↓
HOME / OFFICE behavioral evidence
        ↓
abstention + evidence strength
```

### Một thay đổi quan trọng so với notebook 03 cũ

Notebook cũ dùng một **Beijing reference point + radius** để vừa giới hạn geography, vừa quyết định khi nào được dùng `Asia/Shanghai`.

Sau khi review lại, hai việc này không nên bị gộp chung.

Ở bản này:

```text
timezone
→ xác định trực tiếp từ coordinate của từng stay

geography restriction
→ không áp dụng nếu đề bài không yêu cầu Beijing-only
```

`Asia/Shanghai` chỉ là **tên IANA timezone** được dùng cho China Standard Time / Beijing Time. Tên này **không có nghĩa scope của model là thành phố Shanghai**.

> GeoLife không có direct HOME/OFFICE ground truth. Vì vậy notebook đánh giá **coverage, stability, support và plausibility**, không báo accuracy.

> HOME/OFFICE là sensitive derived locations. Không commit precise user-level inferred coordinates hoặc private caches vào repo.

Design contract: `docs/design/03_home_office_baseline_contract.md`.


In [ ]:
from pathlib import Path
from time import perf_counter
from zoneinfo import ZoneInfo
from IPython.display import display
import os
import pickle
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, DBSCAN

try:
    from timezonefinder import TimezoneFinder
except ImportError:
    # Pin the notebook dependency so the coordinate->timezone audit is reproducible.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "timezonefinder==9.0.0"],
        check=True,
    )
    from timezonefinder import TimezoneFinder

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get(
    "GEOLIFE_REPO_BRANCH",
    "main",
)
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
VOLUME_ROOT = Path("/mnt/geolife-data")
CACHE_DIR = VOLUME_ROOT / "cache" / "cp2_home_office"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def resolve_data_root():
    env_root = os.environ.get("GEOLIFE_DATA_ROOT")
    candidates = ([Path(env_root)] if env_root else []) + [
        VOLUME_ROOT / "extracted" / "Geolife Trajectories 1.3" / "Data",
        VOLUME_ROOT / "Data",
    ]
    for candidate in candidates:
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    for candidate in sorted(VOLUME_ROOT.glob("**/Data")):
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    raise FileNotFoundError("GeoLife Data folder not found")

def ensure_repo():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

DATA_ROOT = resolve_data_root()
ensure_repo()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from notebooks.eda_core import read_plt
from geolife.geo.distance import haversine_m
from geolife.staypoints import clean_trajectory, detect_staypoints
from geolife.model import HomeOfficeConfig, build_semantic_locations, infer_home_office

BASELINE = {
    "same_second_radius_m": 10.0,
    "max_gap_s": 300.0,
    "hard_speed_guard_kmh": 1200.0,
    "distance_threshold_m": 200.0,
    "min_dwell_s": 1200.0,
}

files = sorted(DATA_ROOT.glob("*/Trajectory/*.plt"))
print("Repo branch:", REPO_BRANCH)
print("Data root:", DATA_ROOT)
print("Trajectory files:", f"{len(files):,}")
print("Cache dir:", CACHE_DIR)
print("Frozen CP1 baseline:", BASELINE)


## 1. Materialize frozen CP1 stays ở cấp user

CP1 full-release audit đã xác nhận **5,821 stays** trên 18,670 files, nhưng cache cũ chủ yếu là per-file summary. CP2 cần actual stay rows để gom history theo user.

### Vì sao phải materialize lại actual stays?

Home/Office cần các field mà summary per-file không đủ:

- `user_id`;
- arrival / departure UTC;
- duration;
- stay representative coordinate;
- source-file lineage.

### Reproducibility gate

Section này phải kết thúc bằng:

```text
len(stays) == 5,821
```

Nếu không reconcile đúng CP1 total thì phải dừng — semantic stage không được âm thầm chạy trên một upstream dataset khác.

### Cache design

Run đầu có thể chậm vì phải đọc 18,670 `.plt` files. Notebook checkpoint mỗi 500 files và lưu final private cache bằng pandas pickle để không phụ thuộc `pyarrow`.

Cache chỉ là execution artifact trên mounted volume, không phải dataset để commit.

In [ ]:
STAYS_CACHE = CACHE_DIR / "stays_baseline_v1.pkl"
PARTIAL_CACHE = CACHE_DIR / "stays_baseline_v1.partial.pkl"
EXPECTED_CP1_STAYS = 5821

def user_id_from_path(path):
    return path.parent.parent.name

def process_file(path):
    raw = read_plt(path)[["timestamp", "latitude", "longitude"]]
    cleaned = clean_trajectory(
        raw,
        same_second_radius_m=BASELINE["same_second_radius_m"],
        max_gap_s=BASELINE["max_gap_s"],
        hard_speed_guard_kmh=BASELINE["hard_speed_guard_kmh"],
    )
    stays = detect_staypoints(
        cleaned,
        distance_threshold_m=BASELINE["distance_threshold_m"],
        min_dwell_s=BASELINE["min_dwell_s"],
    )
    if stays.empty:
        return []
    user_id = user_id_from_path(path)
    out = []
    for row in stays.itertuples(index=False):
        out.append({
            "user_id": user_id,
            "source_file": str(path),
            "sequence_id": int(row.sequence_id),
            "arrival_time_utc": row.arrival_time,
            "departure_time_utc": row.departure_time,
            "duration_s": float(row.duration_s),
            "latitude": float(row.latitude),
            "longitude": float(row.longitude),
            "n_points": int(row.n_points),
        })
    return out

if STAYS_CACHE.exists():
    stays = pd.read_pickle(STAYS_CACHE)
    print("Loaded:", STAYS_CACHE)
else:
    if PARTIAL_CACHE.exists():
        with PARTIAL_CACHE.open("rb") as f:
            partial = pickle.load(f)
        processed = set(partial["processed_files"])
        rows = list(partial["rows"])
        print("Resuming partial:", f"{len(processed):,}/{len(files):,} files")
    else:
        processed = set()
        rows = []

    t0 = perf_counter()
    completed_this_run = 0

    for path in files:
        key = str(path)
        if key in processed:
            continue

        rows.extend(process_file(path))
        processed.add(key)
        completed_this_run += 1

        if completed_this_run % 500 == 0:
            elapsed_min = (perf_counter() - t0) / 60
            overall_done = len(processed)
            rate = completed_this_run / max(elapsed_min, 1e-9)
            remaining = len(files) - overall_done
            eta_min = remaining / max(rate, 1e-9)
            print(
                f"{overall_done:,}/{len(files):,} files | "
                f"{len(rows):,} stays | "
                f"elapsed {elapsed_min:.1f} min | ETA ~{eta_min:.1f} min"
            )
            with PARTIAL_CACHE.open("wb") as f:
                pickle.dump(
                    {"processed_files": sorted(processed), "rows": rows},
                    f,
                    protocol=pickle.HIGHEST_PROTOCOL,
                )

    stays = pd.DataFrame(rows)
    stays["arrival_time_utc"] = pd.to_datetime(stays["arrival_time_utc"], utc=True)
    stays["departure_time_utc"] = pd.to_datetime(stays["departure_time_utc"], utc=True)
    stays = stays.sort_values(
        ["user_id", "arrival_time_utc", "source_file"], kind="stable"
    ).reset_index(drop=True)
    stays.to_pickle(STAYS_CACHE)
    if PARTIAL_CACHE.exists():
        PARTIAL_CACHE.unlink()
    print("Saved:", STAYS_CACHE)

print("Materialized stays:", f"{len(stays):,}")
print("Users with stays:", stays["user_id"].nunique())
assert len(stays) == EXPECTED_CP1_STAYS, (
    f"Expected {EXPECTED_CP1_STAYS} CP1 stays, got {len(stays)}"
)
display(stays.head())

### Kết luận phần 1

Full run đã reproduce chính xác:

- **5,821 stays**;
- **136 users** có ít nhất một stay.

Điều này đóng contract giữa CP1 và CP2: mọi semantic analysis về sau bắt đầu từ đúng production stay behavior đã validate.

**Không được suy ra:** 136 users có stays không có nghĩa 136 users đủ evidence để infer HOME/OFFICE. History sufficiency là gate riêng ở phần tiếp theo.

## 2. Kiểm tra mỗi user có đủ lịch sử để suy ra Home/Office hay chưa

### Vì sao cần bước này?

Home và Office là những nơi user thường **quay lại nhiều lần**.

Nếu một user chỉ có 1–2 stays, ta chưa có đủ bằng chứng để nói location nào là Home hay Office.

Ví dụ:

```text
User A:
- 1 stay ở một quán cà phê

User B:
- 20 stays lặp lại ở cùng một khu vực
```

Với User A, nếu vẫn ép model phải trả HOME/OFFICE thì rất dễ tạo ra nhãn sai.

Vì vậy trước khi làm semantic inference, ta cần kiểm tra:

> Mỗi user có bao nhiêu dữ liệu lịch sử và mức support có đủ mạnh hay không?

---

### Section này đo những gì?

Với mỗi user, ta tính:

* `stays`: tổng số stay đã phát hiện;
* `active_utc_dates`: stay xuất hiện trên bao nhiêu ngày khác nhau;
* `first_stay_utc`, `last_stay_utc`: stay đầu tiên và cuối cùng;
* `observation_span_days`: khoảng thời gian từ stay đầu đến stay cuối;
* `total_dwell_h`: tổng thời gian user ở trong các stays;
* `median_stay_min`: thời lượng stay điển hình.

Ví dụ:

```text
stays = 10
active_utc_dates = 6
observation_span_days = 30
```

nghĩa là user có 10 stays, trải trên 6 ngày khác nhau, trong khoảng 30 ngày quan sát.

---

### Vì sao cần cả `stays` và `active_utc_dates`?

Hai metric này đo hai khía cạnh khác nhau.

Ví dụ:

```text
User A:
10 stays nhưng tất cả trong cùng 1 ngày

User B:
10 stays trải trên 8 ngày
```

Cả hai đều có `10 stays`, nhưng User B cho thấy behavior lặp lại theo thời gian rõ hơn.

Vì Home/Office là hành vi lặp lại, số ngày có dữ liệu cũng quan trọng chứ không chỉ số lượng stays.

---

### Kết quả trên GeoLife

Measured support:

| history condition        |   users |
| ------------------------ | ------: |
| >=1 stay                 | **136** |
| >=2 stays                | **120** |
| >=5 stays                |  **99** |
| >=10 stays               |  **81** |
| stays trên >=2 UTC dates | **114** |
| >=5 UTC dates            |  **83** |
| >=10 UTC dates           |  **62** |

GeoLife có tổng cộng **182 users**.

Nhưng chỉ:

```text
136 / 182 users
```

có ít nhất một detected stay.

Tức là:

```text
182 - 136 = 46 users
```

không có stay nào dưới CP1 baseline.

Ngay trong 136 users có stay, mức độ dữ liệu cũng rất khác nhau:

```text
136 users có >=1 stay
120 users có >=2 stays
99 users có >=5 stays
81 users có >=10 stays
```

Nghĩa là càng yêu cầu history mạnh hơn thì số user đủ điều kiện càng giảm.

---

### Tại sao đây dẫn đến `abstention`?

Ta không muốn pipeline hoạt động theo kiểu:

```text
mọi user
→ bắt buộc phải có HOME
→ bắt buộc phải có OFFICE
```

Thay vào đó:

```text
đủ evidence
→ emit HOME/OFFICE

không đủ evidence
→ abstain
```

`abstain` nghĩa là:

> model chủ động không đưa ra nhãn vì dữ liệu hiện tại chưa đủ mạnh.

Đây là behavior mong muốn, không phải lỗi.

---

### Lưu ý về UTC dates

Ở section này, `active_utc_dates` chỉ dùng để đo **độ phủ lịch sử theo ngày**.

Ta chưa dùng nó để suy ra:

```text
ban đêm
giờ làm việc
weekday / weekend
```

vì các khái niệm đó phụ thuộc vào **local timezone**.

Timezone behavioral chỉ được áp dụng sau geography/timezone gate ở các section tiếp theo.


In [ ]:
user_history = (
    stays.assign(
        arrival_utc_date=stays["arrival_time_utc"].dt.date,
    )
    .groupby("user_id")
    .agg(
        stays=("user_id", "size"),
        active_utc_dates=("arrival_utc_date", "nunique"),
        first_stay_utc=("arrival_time_utc", "min"),
        last_stay_utc=("departure_time_utc", "max"),
        total_dwell_h=("duration_s", lambda s: s.sum() / 3600.0),
        median_stay_min=("duration_s", lambda s: s.median() / 60.0),
    )
)

user_history["observation_span_days"] = (
    user_history["last_stay_utc"] - user_history["first_stay_utc"]
).dt.total_seconds() / 86400.0

display(
    user_history[
        ["stays", "active_utc_dates", "observation_span_days", "total_dwell_h", "median_stay_min"]
    ].describe(percentiles=[.1,.25,.5,.75,.9,.95,.99])
)

print("Users with >=1 stay:", len(user_history))
for n in [2, 3, 5, 10]:
    print(f"Users with >= {n} stays:", int((user_history["stays"] >= n).sum()))
for n in [2, 3, 5, 10]:
    print(
        f"Users with stays on >= {n} distinct UTC dates:",
        int((user_history["active_utc_dates"] >= n).sum()),
    )

### Kết luận phần 2

Measured support:

| history condition | users |
|---|---:|
| >=1 stay | **136** |
| >=2 stays | **120** |
| >=5 stays | **99** |
| >=10 stays | **81** |
| stays trên >=2 UTC dates | **114** |
| >=5 UTC dates | **83** |
| >=10 UTC dates | **62** |

Release có 182 users, tức **46 users không có detected stay** dưới frozen CP1 baseline. Ngay trong 136 users còn lại, repeated-history support cũng không đồng đều.

Section này cho thấy dữ liệu GeoLife không đồng đều giữa các user.

Một số user có lịch sử rất dày, nhưng một số khác chỉ có vài stays hoặc không có stay nào.

Vì vậy CP2 phải hỗ trợ:

```text
đủ dữ liệu → infer
thiếu dữ liệu → abstain
```

Quan trọng:

> Coverage cao không đồng nghĩa với semantic certainty cao. Không nên ép HOME/OFFICE cho user chỉ để tăng coverage.

**Decision:** CP2 phải có abstention; coverage của pipeline không được đánh đồng với semantic certainty.

## 3. Thử gom các stays lặp lại thành location bằng DBSCAN

Một stay chỉ nói rằng:

> user đã dừng ở một vị trí trong một khoảng thời gian.

Nhưng Home/Office không nên được suy ra từ **một stay đơn lẻ**.  
Ta cần tìm những nơi user quay lại nhiều lần.

Ví dụ:

```text
stay 1  → gần cùng một khu vực
stay 2  → gần cùng một khu vực
stay 3  → gần cùng một khu vực
```

Ba stays này có thể được gom thành một **recurring location**.

---

### Vì sao thử DBSCAN?

Prototype đầu tiên dùng:

```text
DBSCAN
eps = 200 m
min_samples = 1
```

vì:

* không cần biết trước mỗi user có bao nhiêu locations;
* có thể cluster trực tiếp bằng Haversine distance trên latitude/longitude;
* `200 m` là spatial scale đã dùng ở CP1 nên dễ bắt đầu thử nghiệm.

Clustering được làm **riêng cho từng user**, vì location của user A và user B không liên quan trực tiếp với nhau.

---

### `eps = 200 m` thực sự có nghĩa gì?

Điểm rất dễ hiểu nhầm là:

```text
eps = 200 m
```

**không có nghĩa toàn bộ cluster phải nằm gọn trong đường kính 200 m.**

DBSCAN chỉ yêu cầu các points có thể nối với nhau qua các bước gần nhau.

Ví dụ:

```text
A --180m-- B --180m-- C --180m-- D
```

Mỗi cặp hàng xóm đều cách nhau dưới 200 m:

```text
A ↔ B < 200 m
B ↔ C < 200 m
C ↔ D < 200 m
```

nên DBSCAN có thể đưa tất cả vào cùng một cluster.

Nhưng:

```text
A ↔ D
```

có thể xa hơn 200 m rất nhiều.

Hiện tượng này gọi là **chaining**.

---

In [ ]:
EARTH_RADIUS_M = 6_371_008.8
LOCATION_EPS_M = 200.0

def cluster_user_stays(group, eps_m=LOCATION_EPS_M):
    g = group.sort_values("arrival_time_utc", kind="stable").copy()
    coords_rad = np.radians(g[["latitude", "longitude"]].to_numpy(dtype=float))
    labels = DBSCAN(
        eps=eps_m / EARTH_RADIUS_M,
        min_samples=1,
        metric="haversine",
        algorithm="ball_tree",
    ).fit_predict(coords_rad)
    g["location_id"] = labels.astype(int)
    return g

cluster_parts = []
for user_id, group in stays.groupby("user_id", sort=True):
    clustered_user = cluster_user_stays(group)
    cluster_parts.append(clustered_user)

clustered = pd.concat(cluster_parts, ignore_index=True) if cluster_parts else stays.copy()

location_rows = []
for (user_id, location_id), g in clustered.groupby(["user_id", "location_id"], sort=True):
    lat = float(g["latitude"].median())
    lon = float(g["longitude"].median())
    radii = np.asarray(
        haversine_m(
            g["latitude"].to_numpy(dtype=float),
            g["longitude"].to_numpy(dtype=float),
            lat,
            lon,
        ),
        dtype=float,
    )
    location_rows.append({
        "user_id": user_id,
        "location_id": int(location_id),
        "latitude": lat,
        "longitude": lon,
        "stay_count": len(g),
        "active_utc_dates": g["arrival_time_utc"].dt.date.nunique(),
        "total_dwell_h": g["duration_s"].sum() / 3600.0,
        "max_radius_m": float(np.max(radii)) if len(radii) else 0.0,
    })

locations = pd.DataFrame(location_rows)

print("Users:", locations["user_id"].nunique())
print("Candidate locations:", len(locations))
print("Recurring locations (>=2 stays):", int((locations["stay_count"] >= 2).sum()))
print("Users with >=1 recurring location:", locations.loc[
    locations["stay_count"] >= 2, "user_id"
].nunique())

display(
    locations[
        ["stay_count", "active_utc_dates", "total_dwell_h", "max_radius_m"]
    ].describe(percentiles=[.5,.75,.9,.95,.99])
)

display(
    locations.sort_values("max_radius_m", ascending=False).head(20)
)

### Kết quả prototype `eps=200 m`

Prototype DBSCAN đầu tiên chạy trên toàn bộ 5,821 stays cho thấy:

```text
136 users có stays
1,885 candidate locations
635 recurring locations (>=2 stays)
104 users có ít nhất một recurring location
```

Điều này xác nhận rằng dataset có đủ repeated-location structure để tiếp tục bài toán Home/Office.

Nhưng output cũng phát hiện một điểm rất quan trọng:

```text
DBSCAN eps = 200 m
≠
cluster diameter <= 200 m
```

Cluster rộng nhất có `max_radius_m ≈ 526.7 m`.

Tại sao? DBSCAN chỉ yêu cầu các point có thể nối với nhau qua chuỗi các neighbors gần nhau. Đây là **chaining**.

**Kết luận:** DBSCAN rất hữu ích làm benchmark, nhưng `eps=200m` không thể được diễn giải là “location luôn rộng tối đa 200m”. Vì vậy phía sau ta vẫn cần sensitivity và complete-link comparison.


### Tóm tắt trước khi xử lý timezone

Đến đây ta biết:

```text
5,821 detected stays
136 users có ít nhất một stay
104 users có recurring location trong prototype DBSCAN 200m
```

Nhưng Home/Office không chỉ là bài toán spatial clustering.

Ví dụ một stay xảy ra lúc `13:30 UTC`. Nếu ở China thì local time khoảng `21:30`; nếu ở Los Angeles thì local hour hoàn toàn khác.

Vì vậy trước khi nói đến “ban đêm” hay “giờ làm việc”, ta phải biết **local timezone của từng stay**.

Notebook cũ giải quyết việc này bằng Beijing radius. Bản mới dùng trực tiếp `(latitude, longitude)` của từng stay để tìm timezone, nên không cần giả định toàn bộ user phải thuộc Beijing hay Shanghai.


## 4. Xác định local timezone cho từng stay

GeoLife lưu timestamp theo UTC/GMT, nhưng HOME/OFFICE là behavior theo **local clock**.

Ví dụ cùng một timestamp `13:00 UTC` có thể là:

```text
China        → 21:00 local
Tokyo        → 22:00 local
Los Angeles  → khoảng 05:00 local
```

Do đó không thể dùng một phép cộng cố định cho toàn bộ release.

### Cách làm

Mỗi stay đã có `latitude` và `longitude`. Ta hỏi coordinate này thuộc timezone nào và nhận về IANA timezone id, ví dụ:

```text
Asia/Shanghai
Asia/Tokyo
America/Los_Angeles
Europe/Paris
```

Sau đó timestamp UTC của **chính stay đó** mới được convert sang local time.

```text
stay coordinate
      ↓
timezonefinder
      ↓
IANA timezone_id
      ↓
ZoneInfo(timezone_id)
      ↓
local arrival / departure
```

### `Asia/Shanghai` có nghĩa là gì?

Đây là **tên kỹ thuật của IANA timezone**, không phải filter “chỉ lấy user ở Shanghai”.

Trong GeoLife, phần lớn stays rơi vào `Asia/Shanghai` vì phần lớn dữ liệu nằm ở Trung Quốc.

Notebook có thể đo mức tập trung đó để hiểu dataset, nhưng **không dùng nó để loại user**, vì đề bài hiện tại không yêu cầu Beijing-only hay China-only.


In [ ]:
spatial_summary = stays[["latitude", "longitude"]].describe(
    percentiles=[.01,.05,.25,.5,.75,.95,.99]
)
display(spatial_summary)

user_centers = (
    stays.groupby("user_id")[["latitude", "longitude"]]
    .median()
    .rename(columns={"latitude":"median_latitude","longitude":"median_longitude"})
)
display(user_centers.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

sample_n = min(5000, len(stays))
plot_sample = stays.sample(sample_n, random_state=42) if sample_n else stays
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(plot_sample["longitude"], plot_sample["latitude"], s=8, alpha=0.35)
ax.set(
    title="Stay-point spatial coverage (sample; UTC semantics not yet converted)",
    xlabel="longitude",
    ylabel="latitude",
)
plt.show()

print("NEXT: assign an IANA timezone to every stay from its coordinate.")
print("Behavioral HOME/OFFICE time windows will be applied only after local-time conversion.")


### Kết quả spatial context nói gì?

Output cho thấy stay coordinates tập trung rất mạnh quanh khu vực Beijing/China:

```text
median latitude  ≈ 39.98
median longitude ≈ 116.33
```

nhưng tail của dữ liệu đi rất xa.

Điều này giải thích vì sao **không nên gán cùng một local clock cho toàn bộ dataset**.

Nhưng spatial distribution này chỉ là context. Ta **không** dùng distance tới một Beijing point để quyết định timezone nữa.


### 4.1 Gán IANA timezone cho từng stay từ latitude / longitude

Thay vì hỏi:

```text
stay có cách Beijing center <= 100 km không?
```

ta hỏi trực tiếp:

```text
(lat, lon) của stay này
nằm trong timezone nào?
```

Ví dụ:

```text
39.98, 116.33 → Asia/Shanghai
35.68, 139.76 → Asia/Tokyo
34.05, -118.24 → America/Los_Angeles
```

### Vì sao cách này dễ hiểu hơn?

Timezone boundary không phải một rectangle đơn giản và cũng không phải bán kính X km quanh một thành phố.

`timezonefinder` thực hiện coordinate lookup trên timezone-boundary data và trả về IANA timezone id.

### Ta còn đo `Asia/Shanghai` share để làm gì?

Chỉ để **mô tả dataset**.

Ta tính:

```text
reference_tz_stay_share
reference_tz_dwell_share
```

rồi thử `50% / 80% / 90% / 95%`.

Nhưng bảng này **không còn là eligibility gate**.

Ví dụ một user chủ yếu ở Paris vẫn được giữ nếu các stay có timezone xác định được và phía sau có đủ recurring / behavioral evidence.

### Kết luận

```text
timezone id
→ dùng để tính local time

Asia/Shanghai share
→ dùng để hiểu dataset

không dùng timezone name
→ để giả định user đang ở Beijing hoặc Shanghai
```


In [ ]:
REFERENCE_TZ_FOR_PROFILE = "Asia/Shanghai"
REFERENCE_SHARE_VALUES = [0.50, 0.80, 0.90, 0.95]

timezone_finder = TimezoneFinder()

stays_tz = stays.copy()

stays_tz["timezone_id"] = [
    timezone_finder.timezone_at(
        lng=float(lon),
        lat=float(lat),
    )
    for lat, lon in zip(
        stays_tz["latitude"].to_numpy(dtype=float),
        stays_tz["longitude"].to_numpy(dtype=float),
    )
]

unresolved_tz = int(stays_tz["timezone_id"].isna().sum())

print("Timezone lookup:")
print("  stays:", len(stays_tz))
print("  resolved timezone:", len(stays_tz) - unresolved_tz)
print("  unresolved timezone:", unresolved_tz)

timezone_summary = (
    stays_tz.assign(dwell_h=stays_tz["duration_s"] / 3600.0)
    .groupby("timezone_id", dropna=False)
    .agg(
        stays=("user_id", "size"),
        users=("user_id", "nunique"),
        dwell_h=("dwell_h", "sum"),
    )
    .sort_values(["stays", "dwell_h"], ascending=False)
)

print("\nTop timezone IDs by stay count:")
display(timezone_summary.head(20))

stays_tz["in_reference_timezone"] = (
    stays_tz["timezone_id"] == REFERENCE_TZ_FOR_PROFILE
)
stays_tz["reference_timezone_dwell_s"] = np.where(
    stays_tz["in_reference_timezone"],
    stays_tz["duration_s"],
    0.0,
)

user_timezone_profile = (
    stays_tz.groupby("user_id")
    .agg(
        total_stays=("user_id", "size"),
        reference_tz_stays=("in_reference_timezone", "sum"),
        total_dwell_s=("duration_s", "sum"),
        reference_tz_dwell_s=("reference_timezone_dwell_s", "sum"),
        distinct_timezones=("timezone_id", "nunique"),
    )
)

user_timezone_profile["reference_tz_stay_share"] = (
    user_timezone_profile["reference_tz_stays"] / user_timezone_profile["total_stays"]
)
user_timezone_profile["reference_tz_dwell_share"] = np.where(
    user_timezone_profile["total_dwell_s"] > 0,
    user_timezone_profile["reference_tz_dwell_s"] / user_timezone_profile["total_dwell_s"],
    0.0,
)

profile_rows = []
for min_share in REFERENCE_SHARE_VALUES:
    matches = (
        (user_timezone_profile["reference_tz_stay_share"] >= min_share)
        & (user_timezone_profile["reference_tz_dwell_share"] >= min_share)
    )
    profile_rows.append(
        {
            "min_both_shares": min_share,
            "users_matching_profile": int(matches.sum()),
            "user_rate": float(matches.mean()),
        }
    )

timezone_profile_sensitivity = pd.DataFrame(profile_rows)

print("\nAsia/Shanghai concentration diagnostic (NOT a filtering rule):")
display(
    timezone_profile_sensitivity.style.format(
        {"min_both_shares": "{:.0%}", "user_rate": "{:.2%}"}
    )
)

stays_semantic = stays_tz[
    stays_tz["timezone_id"].notna()
].copy()

print("\nFinal timezone-resolved semantic scope:")
print("Users retained:", stays_semantic["user_id"].nunique(), "/", stays["user_id"].nunique())
print("Stays retained:", len(stays_semantic), "/", len(stays))
print("Timezone-based user filter: NONE")


### Diễn giải output timezone lookup

Trong run vừa kiểm tra:

```text
5,821 stays
→ 5,821 stays xác định được timezone
→ 0 unresolved
```

Timezone phổ biến nhất là:

```text
Asia/Shanghai
→ 5,599 stays
→ xuất hiện ở 135 / 136 users
```

Các timezone khác gồm America/Los_Angeles, Europe/Paris, Asia/Bangkok, Asia/Hong_Kong, Asia/Tokyo...

Điều này cho thấy GeoLife **rất tập trung ở Trung Quốc nhưng vẫn có travel observations**.

Run trước cho thấy profile:

```text
50% → 134 users
80% → 128 users
90% → 128 users
95% → 126 users
```

Nhưng vì đề bài không yêu cầu China-only / Beijing-only, notebook cuối **không dùng bảng này để loại user**.

Abstention sẽ xảy ra ở các bước hợp lý hơn:

```text
thiếu history
không có recurring location
behavioral evidence yếu
```


### Kết luận phần timezone assignment

Sau section này, mỗi stay có thêm `timezone_id`.

Ta **không** biến user thành “Shanghai user” hay “Beijing user”.

Ta chỉ trả lời:

> stay này phải được đọc bằng chiếc đồng hồ địa phương nào?

Đó mới là thông tin cần cho Home/Office.

Nếu sau này product requirement thêm “chỉ infer ở Beijing”, geography gate có thể được thêm riêng mà không phải sửa timezone logic.


### 4.2 Chuyển UTC sang local time của từng stay

Mỗi stay đã có `timezone_id`.

Ta convert:

```text
arrival_time_utc
departure_time_utc
        ↓
ZoneInfo(timezone_id)
        ↓
arrival_time_local
departure_time_local
```

Một DataFrame có thể chứa nhiều timezone, nên notebook giữ `timezone_id` riêng và lưu local timestamps dưới dạng local wall-clock time.

Nhờ vậy phía sau có thể tính local date / weekday / hour ổn định.

> Không cộng tay `+8h`. Mỗi stay dùng timezone của chính coordinate đó.


In [ ]:
def utc_to_local_wall_time(timestamp_utc, timezone_id):
    if pd.isna(timezone_id):
        return pd.NaT
    ts = pd.Timestamp(timestamp_utc)
    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    return ts.tz_convert(ZoneInfo(str(timezone_id))).tz_localize(None)

stays_semantic["arrival_time_local"] = pd.to_datetime(
    [
        utc_to_local_wall_time(ts, tzid)
        for ts, tzid in zip(stays_semantic["arrival_time_utc"], stays_semantic["timezone_id"])
    ]
)
stays_semantic["departure_time_local"] = pd.to_datetime(
    [
        utc_to_local_wall_time(ts, tzid)
        for ts, tzid in zip(stays_semantic["departure_time_utc"], stays_semantic["timezone_id"])
    ]
)

stays_semantic["arrival_local_date"] = stays_semantic["arrival_time_local"].dt.date
stays_semantic["arrival_local_hour"] = stays_semantic["arrival_time_local"].dt.hour
stays_semantic["arrival_local_weekday"] = stays_semantic["arrival_time_local"].dt.weekday

assert stays_semantic["arrival_time_local"].notna().all()
assert stays_semantic["departure_time_local"].notna().all()

print("Users in timezone-resolved semantic scope:", stays_semantic["user_id"].nunique())
print("Stays retained for semantic analysis:", len(stays_semantic))
print("Distinct IANA timezones retained:", stays_semantic["timezone_id"].nunique())

print("\nTimezone mix:")
display(stays_semantic["timezone_id"].value_counts().rename("stays").to_frame().head(20))

print("\nSample local-time conversion (coordinates omitted):")
display(
    stays_semantic[
        ["user_id","timezone_id","arrival_time_utc","arrival_time_local","departure_time_local","duration_s"]
    ].head(10)
)


### 4.3 DBSCAN `eps` sensitivity trên toàn bộ timezone-resolved stays

Ta thử `10 / 20 / 30 / 50 / 100 / 150 / 200m` và đo locations, recurring locations/users, cùng cluster diameter.

```text
eps nhỏ
→ compact hơn nhưng có thể fragment

eps lớn
→ recurrence coverage tăng nhưng chaining có thể tăng
```

Vì semantic scope bản cuối là **all timezone-resolved stays**, bảng này phải được Run All lại; không copy số của cohort `80/80` cũ.


In [ ]:
DBSCAN_EPS_VALUES_M = [10.0, 20.0, 30.0, 50.0, 100.0, 150.0, 200.0]

def summarize_dbscan_eps(stays_input, eps_m):
    location_rows = []

    for user_id, group in stays_input.groupby("user_id", sort=True):
        g = group.sort_values("arrival_time_utc", kind="stable").copy()

        lat = g["latitude"].to_numpy(dtype=float)
        lon = g["longitude"].to_numpy(dtype=float)

        coords_rad = np.radians(
            g[["latitude", "longitude"]].to_numpy(dtype=float)
        )

        labels = DBSCAN(
            eps=eps_m / EARTH_RADIUS_M,
            min_samples=1,
            metric="haversine",
            algorithm="ball_tree",
        ).fit_predict(coords_rad)

        distances = np.asarray(
            haversine_m(
                lat[:, None],
                lon[:, None],
                lat[None, :],
                lon[None, :],
            ),
            dtype=float,
        )

        for location_id in np.unique(labels):
            member_idx = np.flatnonzero(labels == location_id)
            members = g.iloc[member_idx]

            diameter_m = (
                float(distances[np.ix_(member_idx, member_idx)].max())
                if len(member_idx) > 1
                else 0.0
            )

            location_rows.append({
                "user_id": user_id,
                "location_id": int(location_id),
                "stay_count": len(member_idx),
                "active_local_dates": members["arrival_local_date"].nunique(),
                "total_dwell_h": members["duration_s"].sum() / 3600.0,
                "diameter_m": diameter_m,
            })

    locations_eps = pd.DataFrame(location_rows)
    recurring = locations_eps["stay_count"] >= 2
    recurring_locations = locations_eps.loc[recurring].copy()

    summary = {
        "eps_m": eps_m,
        "locations": len(locations_eps),
        "recurring_locations": int(recurring.sum()),
        "users_with_recurring_location": int(
            recurring_locations["user_id"].nunique()
        ),
        "median_locations_per_user": float(
            locations_eps.groupby("user_id").size().median()
        ),
        "median_recurring_diameter_m": float(
            recurring_locations["diameter_m"].median()
        ) if len(recurring_locations) else np.nan,
        "p95_recurring_diameter_m": float(
            recurring_locations["diameter_m"].quantile(0.95)
        ) if len(recurring_locations) else np.nan,
        "max_recurring_diameter_m": float(
            recurring_locations["diameter_m"].max()
        ) if len(recurring_locations) else np.nan,
        "recurring_clusters_le_200_rate": float(
            (recurring_locations["diameter_m"] <= 200.0).mean()
        ) if len(recurring_locations) else np.nan,
        "recurring_clusters_gt_200": int(
            (recurring_locations["diameter_m"] > 200.0).sum()
        ) if len(recurring_locations) else 0,
    }

    return locations_eps, summary


dbscan_sensitivity_rows = []
dbscan_artifacts = {}

for eps_m in DBSCAN_EPS_VALUES_M:
    locations_eps, summary = summarize_dbscan_eps(
        stays_semantic,
        eps_m,
    )
    dbscan_artifacts[eps_m] = locations_eps
    dbscan_sensitivity_rows.append(summary)

dbscan_eps_sensitivity = pd.DataFrame(dbscan_sensitivity_rows)

display(
    dbscan_eps_sensitivity[
        [
            "eps_m",
            "locations",
            "recurring_locations",
            "users_with_recurring_location",
            "median_locations_per_user",
            "median_recurring_diameter_m",
            "p95_recurring_diameter_m",
            "max_recurring_diameter_m",
            "recurring_clusters_le_200_rate",
            "recurring_clusters_gt_200",
        ]
    ]
)


### Cách đọc output DBSCAN sensitivity

Nhìn đồng thời:

```text
users_with_recurring_location
p95 / max recurring diameter
```

Nếu `eps` tăng làm recurring users tăng nhưng diameter tăng rất mạnh, DBSCAN đang đổi **coverage lấy compactness**.

`eps=200m` không đảm bảo `diameter<=200m`; recurring clusters vượt 200m là chaining, không phải bug.


### 4.4 Complete-link recurring-location audit

Complete-link có threshold dễ diễn giải hơn:

```text
threshold = 200m
→ maximum pairwise distance trong cluster <= khoảng 200m
```

Ta thử `100 / 200 / 300m` để xem mức nào giữ recurring-user coverage nhưng vẫn compact.


In [ ]:
COMPLETE_LINK_THRESHOLDS_M = [100.0, 200.0, 300.0]
CANDIDATE_COMPLETE_LINK_M = 200.0

def pairwise_haversine_matrix_m(group):
    lat = group["latitude"].to_numpy(dtype=float)
    lon = group["longitude"].to_numpy(dtype=float)
    return np.asarray(
        haversine_m(
            lat[:, None],
            lon[:, None],
            lat[None, :],
            lon[None, :],
        ),
        dtype=float,
    )

def complete_link_user(group, threshold_m):
    g = group.sort_values("arrival_time_utc", kind="stable").copy()
    n = len(g)
    if n == 1:
        g["location_id"] = 0
        return g, np.zeros((1, 1), dtype=float)

    distances = pairwise_haversine_matrix_m(g)
    labels = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="complete",
        distance_threshold=threshold_m,
    ).fit_predict(distances)
    g["location_id"] = labels.astype(int)
    return g, distances

def summarize_complete_link(threshold_m):
    clustered_parts = []
    location_rows = []

    for user_id, group in stays_semantic.groupby("user_id", sort=True):
        clustered_user, distances = complete_link_user(group, threshold_m)
        clustered_parts.append(clustered_user)

        labels = clustered_user["location_id"].to_numpy(dtype=int)
        for location_id in np.unique(labels):
            member_idx = np.flatnonzero(labels == location_id)
            members = clustered_user.iloc[member_idx]
            diameter_m = (
                float(distances[np.ix_(member_idx, member_idx)].max())
                if len(member_idx) > 1
                else 0.0
            )
            location_rows.append({
                "user_id": user_id,
                "location_id": int(location_id),
                "latitude": float(members["latitude"].median()),
                "longitude": float(members["longitude"].median()),
                "stay_count": len(members),
                "active_local_dates": members["arrival_local_date"].nunique(),
                "total_dwell_h": members["duration_s"].sum() / 3600.0,
                "diameter_m": diameter_m,
            })

    clustered_all = pd.concat(clustered_parts, ignore_index=True)
    locations_all = pd.DataFrame(location_rows)

    recurring = locations_all["stay_count"] >= 2
    return clustered_all, locations_all, {
        "threshold_m": threshold_m,
        "locations": len(locations_all),
        "recurring_locations": int(recurring.sum()),
        "users_with_recurring_location": int(
            locations_all.loc[recurring, "user_id"].nunique()
        ),
        "median_locations_per_user": float(
            locations_all.groupby("user_id").size().median()
        ),
        "p95_diameter_m": float(locations_all["diameter_m"].quantile(0.95)),
        "max_diameter_m": float(locations_all["diameter_m"].max()),
    }

cluster_sensitivity_rows = []
cluster_artifacts = {}

for threshold_m in COMPLETE_LINK_THRESHOLDS_M:
    clustered_threshold, locations_threshold, summary = summarize_complete_link(
        threshold_m
    )
    cluster_sensitivity_rows.append(summary)
    cluster_artifacts[threshold_m] = (
        clustered_threshold,
        locations_threshold,
    )

complete_link_sensitivity = pd.DataFrame(cluster_sensitivity_rows)
display(complete_link_sensitivity)

semantic_stays, semantic_locations = cluster_artifacts[CANDIDATE_COMPLETE_LINK_M]

assert semantic_locations["diameter_m"].max() <= CANDIDATE_COMPLETE_LINK_M + 1e-6

print("Candidate complete-link threshold:", CANDIDATE_COMPLETE_LINK_M, "m")
print("Semantic locations:", len(semantic_locations))
print(
    "Recurring semantic locations (>=2 stays):",
    int((semantic_locations["stay_count"] >= 2).sum()),
)
print(
    "Users with recurring semantic location:",
    semantic_locations.loc[
        semantic_locations["stay_count"] >= 2, "user_id"
    ].nunique(),
)
print(
    "Max verified cluster diameter (m):",
    semantic_locations["diameter_m"].max(),
)

display(
    semantic_locations.sort_values(
        ["stay_count", "total_dwell_h"],
        ascending=False,
    ).head(30)
)


### Cách đọc output complete-link

Ở notebook cũ, `200m` là mức cân bằng tốt trên Beijing-focused cohort. Bản cuối dùng scope rộng hơn nên phải rerun.

Đọc theo pattern:

```text
100 → 200m:
nếu recurring users tăng rõ → 100m có thể fragment

200 → 300m:
nếu recurring users gần như không tăng nhưng diameter tăng
→ 300m chỉ làm location rộng hơn
```

Sanity check: `max_diameter_m <= threshold_m` phải đúng với complete-link.


### 4.5 So sánh DBSCAN và complete-link

DBSCAN dùng `30 / 100 / 200m` làm representative settings; complete-link dùng `200m`.

So sánh bằng coverage + compactness, không phải supervised accuracy.

Câu hỏi là:

> representation nào giữ repeated-location coverage tốt mà vẫn có spatial semantics dễ kiểm soát?


In [ ]:
dbscan_compare = (
    dbscan_eps_sensitivity[
        dbscan_eps_sensitivity["eps_m"].isin([30.0, 100.0, 200.0])
    ][
        [
            "eps_m",
            "locations",
            "recurring_locations",
            "users_with_recurring_location",
            "p95_recurring_diameter_m",
            "max_recurring_diameter_m",
            "recurring_clusters_le_200_rate",
        ]
    ]
    .copy()
)

dbscan_compare["method"] = "DBSCAN"
dbscan_compare["setting_m"] = dbscan_compare["eps_m"]
dbscan_compare = dbscan_compare.rename(
    columns={
        "p95_recurring_diameter_m": "p95_diameter_m",
        "max_recurring_diameter_m": "max_diameter_m",
        "recurring_clusters_le_200_rate": "clusters_le_200_rate",
    }
)

complete_compare = complete_link_sensitivity[
    complete_link_sensitivity["threshold_m"] == CANDIDATE_COMPLETE_LINK_M
].copy()

complete_compare["method"] = "complete-link"
complete_compare["setting_m"] = complete_compare["threshold_m"]
complete_compare["clusters_le_200_rate"] = (
    complete_compare["max_diameter_m"] <= CANDIDATE_COMPLETE_LINK_M + 1e-6
).astype(float)

clustering_benchmark = pd.concat(
    [
        dbscan_compare[
            [
                "method",
                "setting_m",
                "locations",
                "recurring_locations",
                "users_with_recurring_location",
                "p95_diameter_m",
                "max_diameter_m",
                "clusters_le_200_rate",
            ]
        ],
        complete_compare[
            [
                "method",
                "setting_m",
                "locations",
                "recurring_locations",
                "users_with_recurring_location",
                "p95_diameter_m",
                "max_diameter_m",
                "clusters_le_200_rate",
            ]
        ],
    ],
    ignore_index=True,
)

display(clustering_benchmark)


### Diễn giải benchmark clustering

Ưu tiên hai câu hỏi:

1. recurring-user coverage có khác nhiều không?
2. cluster rộng tới đâu?

Complete-link `200m` có lợi thế engineering là hard diameter contract. DBSCAN vẫn được giữ làm benchmark để quan sát trade-off density connectivity.

Decision không phải “DBSCAN sai”, mà là representation nào dễ giải thích hơn cho recurring place.


## 5. Timezone-aware semantic configuration

Semantic scope:

```text
mọi CP1 stay có timezone lookup được
```

Không có Beijing radius gate, `Asia/Shanghai 80/80` gate hay Shanghai-city gate.

Timezone:

```text
(lat, lon) → timezone_id → ZoneInfo → local time
```

Complete-link `200m` và scoring gates v1 được giữ làm **working controls**, rồi phải audit lại sau Run All trên scope mới.

HOME: `21–06 local / 3 dates / 0.50 share / 0.20 margin`.

OFFICE: `Mon–Fri 09–17 local / 3 dates / 0.30 share / 0.10 margin`.

Evidence yếu → abstain.


In [ ]:
TIMEZONE_AWARE_CONFIG = {
    "timezone_assignment": "coordinate_polygon_lookup",
    "timezone_lookup_library": "timezonefinder==9.0.0",
    "timezone_user_filter": None,
    "reference_timezone_for_dataset_profile": REFERENCE_TZ_FOR_PROFILE,
    "retain_all_resolved_timezone_stays": True,
    "clustering_method_control": "complete_link",
    "candidate_location_complete_link_m": CANDIDATE_COMPLETE_LINK_M,
    "home_night_start_hour": 21,
    "home_night_end_hour": 6,
    "office_start_hour": 9,
    "office_end_hour": 17,
    "office_weekdays": [0,1,2,3,4],
    "home_min_dates": 3,
    "home_min_share": 0.50,
    "home_min_margin": 0.20,
    "office_min_dates": 3,
    "office_min_share": 0.30,
    "office_min_margin": 0.10,
    "support_saturation_dates": 5,
}
display(pd.Series(TIMEZONE_AWARE_CONFIG, name="working_value"))
print("STATUS: final notebook semantics are timezone-aware and geography-agnostic.")
print("Production src still reflects CP2 v1 until migration is implemented and tested.")


## 6. Home / Office scoring audit

HOME/OFFICE được tính trên **local time của từng stay**.

Một stay `20:50–21:30` chỉ đóng góp `21:00–21:30` vào HOME window, nên notebook dùng exact interval overlap thay vì chỉ nhìn arrival hour.

HOME: `21:00–06:00 local`.

OFFICE: `Mon–Fri 09:00–17:00 local`.

Mỗi recurring location được đo bằng overlap dwell, dwell share, relevant dates và margin so với runner-up.

HOME/OFFICE không bị ép phải là hai locations khác nhau; ambiguity được giữ để audit.


In [ ]:
HOME_NIGHT_START_HOUR = 21
HOME_NIGHT_END_HOUR = 6
OFFICE_START_HOUR = 9
OFFICE_END_HOUR = 17
OFFICE_WEEKDAYS = {0, 1, 2, 3, 4}

MIN_RELEVANT_DATE_OVERLAP_S = 10 * 60
MIN_RELEVANT_DATES = 2

def interval_overlap_s(start, end, window_start, window_end):
    overlap_start = max(start, window_start)
    overlap_end = min(end, window_end)
    if overlap_end <= overlap_start:
        return 0.0
    return float((overlap_end - overlap_start).total_seconds())

def stay_window_contributions(row):
    start = row.arrival_time_local
    end = row.departure_time_local

    night_rows = []
    office_rows = []

    # A stay shortly after midnight can overlap the night window that started
    # on the previous local date, so include one padded date before arrival.
    day = start.normalize() - pd.Timedelta(days=1)
    last_day = end.normalize()

    while day <= last_day:
        night_start = day + pd.Timedelta(hours=HOME_NIGHT_START_HOUR)
        night_end = day + pd.Timedelta(days=1, hours=HOME_NIGHT_END_HOUR)
        night_s = interval_overlap_s(start, end, night_start, night_end)
        if night_s > 0:
            night_rows.append(
                {
                    "user_id": row.user_id,
                    "location_id": int(row.location_id),
                    "behavior_date": night_start.date(),
                    "overlap_s": night_s,
                }
            )

        if day.weekday() in OFFICE_WEEKDAYS:
            office_start = day + pd.Timedelta(hours=OFFICE_START_HOUR)
            office_end = day + pd.Timedelta(hours=OFFICE_END_HOUR)
            office_s = interval_overlap_s(start, end, office_start, office_end)
            if office_s > 0:
                office_rows.append(
                    {
                        "user_id": row.user_id,
                        "location_id": int(row.location_id),
                        "behavior_date": office_start.date(),
                        "overlap_s": office_s,
                    }
                )

        day += pd.Timedelta(days=1)

    return night_rows, office_rows

night_rows = []
office_rows = []

for row in semantic_stays.itertuples(index=False):
    nr, orows = stay_window_contributions(row)
    night_rows.extend(nr)
    office_rows.extend(orows)

night_contrib = pd.DataFrame(
    night_rows,
    columns=["user_id", "location_id", "behavior_date", "overlap_s"],
)
office_contrib = pd.DataFrame(
    office_rows,
    columns=["user_id", "location_id", "behavior_date", "overlap_s"],
)

def aggregate_relevant_window(contrib, prefix):
    if contrib.empty:
        return pd.DataFrame(
            columns=[
                "user_id",
                "location_id",
                f"{prefix}_dwell_s",
                f"{prefix}_dates",
            ]
        )

    per_date = (
        contrib.groupby(["user_id", "location_id", "behavior_date"], as_index=False)
        ["overlap_s"]
        .sum()
    )

    dwell = (
        per_date.groupby(["user_id", "location_id"], as_index=False)["overlap_s"]
        .sum()
        .rename(columns={"overlap_s": f"{prefix}_dwell_s"})
    )

    supported_dates = (
        per_date.loc[per_date["overlap_s"] >= MIN_RELEVANT_DATE_OVERLAP_S]
        .groupby(["user_id", "location_id"], as_index=False)["behavior_date"]
        .nunique()
        .rename(columns={"behavior_date": f"{prefix}_dates"})
    )

    return dwell.merge(
        supported_dates,
        on=["user_id", "location_id"],
        how="left",
    ).fillna({f"{prefix}_dates": 0})

night_features = aggregate_relevant_window(night_contrib, "night")
office_features = aggregate_relevant_window(office_contrib, "office")

semantic_features = (
    semantic_locations.merge(
        night_features,
        on=["user_id", "location_id"],
        how="left",
    )
    .merge(
        office_features,
        on=["user_id", "location_id"],
        how="left",
    )
)

for col in ["night_dwell_s", "night_dates", "office_dwell_s", "office_dates"]:
    semantic_features[col] = semantic_features[col].fillna(0)

semantic_features["night_dates"] = semantic_features["night_dates"].astype(int)
semantic_features["office_dates"] = semantic_features["office_dates"].astype(int)

user_night_total = semantic_features.groupby("user_id")["night_dwell_s"].transform("sum")
user_office_total = semantic_features.groupby("user_id")["office_dwell_s"].transform("sum")

semantic_features["night_dwell_share"] = np.where(
    user_night_total > 0,
    semantic_features["night_dwell_s"] / user_night_total,
    0.0,
)
semantic_features["office_dwell_share"] = np.where(
    user_office_total > 0,
    semantic_features["office_dwell_s"] / user_office_total,
    0.0,
)

semantic_features["night_dwell_h"] = semantic_features["night_dwell_s"] / 3600.0
semantic_features["office_dwell_h"] = semantic_features["office_dwell_s"] / 3600.0

def rank_semantic_candidates(
    features,
    *,
    share_col,
    dates_col,
    dwell_col,
    label,
):
    eligible = features[
        (features["stay_count"] >= 2)
        & (features[dates_col] >= MIN_RELEVANT_DATES)
        & (features[dwell_col] > 0)
    ].copy()

    if eligible.empty:
        return eligible

    eligible = eligible.sort_values(
        ["user_id", share_col, dates_col, dwell_col, "stay_count"],
        ascending=[True, False, False, False, False],
        kind="stable",
    )
    eligible[f"{label}_rank"] = eligible.groupby("user_id").cumcount() + 1
    return eligible

home_ranked = rank_semantic_candidates(
    semantic_features,
    share_col="night_dwell_share",
    dates_col="night_dates",
    dwell_col="night_dwell_s",
    label="home",
)
office_ranked = rank_semantic_candidates(
    semantic_features,
    share_col="office_dwell_share",
    dates_col="office_dates",
    dwell_col="office_dwell_s",
    label="office",
)

def top_with_margin(ranked, *, label, share_col):
    if ranked.empty:
        return pd.DataFrame()

    top1 = ranked[ranked[f"{label}_rank"] == 1].copy()
    second = (
        ranked[ranked[f"{label}_rank"] == 2][["user_id", share_col]]
        .rename(columns={share_col: f"{label}_second_share"})
    )
    top1 = top1.merge(second, on="user_id", how="left")
    top1[f"{label}_second_share"] = top1[f"{label}_second_share"].fillna(0.0)
    top1[f"{label}_share_margin"] = (
        top1[share_col] - top1[f"{label}_second_share"]
    )
    return top1

home_top = top_with_margin(
    home_ranked,
    label="home",
    share_col="night_dwell_share",
)
office_top = top_with_margin(
    office_ranked,
    label="office",
    share_col="office_dwell_share",
)

candidate_users = set(stays_semantic["user_id"].unique())
home_users = set(home_top["user_id"]) if not home_top.empty else set()
office_users = set(office_top["user_id"]) if not office_top.empty else set()
both_users = home_users & office_users

print("Timezone-resolved users:", len(candidate_users))
print("Users with recurring semantic location:", semantic_locations.loc[
    semantic_locations["stay_count"] >= 2, "user_id"
].nunique())
print("Users with supported HOME candidate:", len(home_users))
print("Users with supported OFFICE candidate:", len(office_users))
print("Users with both candidates:", len(both_users))
print("Users abstaining from HOME:", len(candidate_users - home_users))
print("Users abstaining from OFFICE:", len(candidate_users - office_users))

if both_users:
    paired = (
        home_top[home_top["user_id"].isin(both_users)][
            ["user_id", "location_id"]
        ]
        .rename(columns={"location_id": "home_location_id"})
        .merge(
            office_top[office_top["user_id"].isin(both_users)][
                ["user_id", "location_id"]
            ].rename(columns={"location_id": "office_location_id"}),
            on="user_id",
        )
    )
    paired["same_location_candidate"] = (
        paired["home_location_id"] == paired["office_location_id"]
    )
    print(
        "Both-candidate users with same leading location:",
        int(paired["same_location_candidate"].sum()),
        "/",
        len(paired),
    )

def candidate_distribution(top, cols):
    if top.empty:
        return pd.DataFrame()
    return top[cols].describe(
        percentiles=[.1, .25, .5, .75, .9, .95]
    )

print("\nHOME top-candidate evidence:")
display(
    candidate_distribution(
        home_top,
        [
            "night_dwell_share",
            "home_share_margin",
            "night_dates",
            "night_dwell_h",
            "stay_count",
        ],
    )
)

print("\nOFFICE top-candidate evidence:")
display(
    candidate_distribution(
        office_top,
        [
            "office_dwell_share",
            "office_share_margin",
            "office_dates",
            "office_dwell_h",
            "stay_count",
        ],
    )
)

print("\nSample HOME candidates (no coordinates displayed):")
display(
    home_top[
        [
            "user_id",
            "location_id",
            "stay_count",
            "active_local_dates",
            "night_dates",
            "night_dwell_h",
            "night_dwell_share",
            "home_share_margin",
        ]
    ].head(20)
)

print("\nSample OFFICE candidates (no coordinates displayed):")
display(
    office_top[
        [
            "user_id",
            "location_id",
            "stay_count",
            "active_local_dates",
            "office_dates",
            "office_dwell_h",
            "office_dwell_share",
            "office_share_margin",
        ]
    ].head(20)
)


### 6.0 Cách đọc output scoring ban đầu

Cell phía trên chưa áp final emission gates. Nó chỉ tạo HOME/OFFICE candidates có recurring location và evidence trên >=2 ngày.

Đọc funnel:

```text
timezone-resolved users
→ recurring-location users
→ HOME candidate users
→ OFFICE candidate users
→ final gates phía sau
```

Giảm coverage ở mỗi bước không tự động là lỗi; có thể đơn giản là evidence chưa đủ mạnh.

`dwell_share` = location chiếm bao nhiêu behavioral evidence.

`margin` = top location hơn runner-up bao nhiêu.

`relevant_dates` = evidence có repeat qua nhiều ngày hay không.


### 6.1 Kiểm tra độ ổn định của scoring

Không có ground truth nên ta kiểm tra stability.

HOME windows: `20–06 / 21–06 / 22–06`.

OFFICE windows: `08–17 / 09–17 / 09–18`.

Nếu phần lớn shared users giữ nguyên top location khi dịch một giờ, heuristic ít phụ thuộc vào một mốc giờ quá cụ thể.

Ta cũng thử nhiều mức dates/share/margin. Rule càng strict → ít emissions hơn → nhiều abstention hơn.

Sau Run All trên scope mới, kiểm tra baseline v1 còn nằm ở vùng ổn định hay không.


In [ ]:
def build_semantic_features_for_windows(
    *,
    home_start_hour,
    home_end_hour,
    office_start_hour,
    office_end_hour,
):
    night_rows = []
    office_rows = []

    for row in semantic_stays.itertuples(index=False):
        start = row.arrival_time_local
        end = row.departure_time_local

        day = start.normalize() - pd.Timedelta(days=1)
        last_day = end.normalize()

        while day <= last_day:
            night_start = day + pd.Timedelta(hours=home_start_hour)
            night_end = day + pd.Timedelta(days=1, hours=home_end_hour)
            night_s = interval_overlap_s(start, end, night_start, night_end)
            if night_s > 0:
                night_rows.append(
                    {
                        "user_id": row.user_id,
                        "location_id": int(row.location_id),
                        "behavior_date": night_start.date(),
                        "overlap_s": night_s,
                    }
                )

            if day.weekday() in OFFICE_WEEKDAYS:
                office_start = day + pd.Timedelta(hours=office_start_hour)
                office_end = day + pd.Timedelta(hours=office_end_hour)
                office_s = interval_overlap_s(start, end, office_start, office_end)
                if office_s > 0:
                    office_rows.append(
                        {
                            "user_id": row.user_id,
                            "location_id": int(row.location_id),
                            "behavior_date": office_start.date(),
                            "overlap_s": office_s,
                        }
                    )

            day += pd.Timedelta(days=1)

    night_contrib_local = pd.DataFrame(
        night_rows,
        columns=["user_id", "location_id", "behavior_date", "overlap_s"],
    )
    office_contrib_local = pd.DataFrame(
        office_rows,
        columns=["user_id", "location_id", "behavior_date", "overlap_s"],
    )

    night_features_local = aggregate_relevant_window(
        night_contrib_local,
        "night",
    )
    office_features_local = aggregate_relevant_window(
        office_contrib_local,
        "office",
    )

    features = (
        semantic_locations.merge(
            night_features_local,
            on=["user_id", "location_id"],
            how="left",
        )
        .merge(
            office_features_local,
            on=["user_id", "location_id"],
            how="left",
        )
    )

    for col in ["night_dwell_s", "night_dates", "office_dwell_s", "office_dates"]:
        features[col] = features[col].fillna(0)

    features["night_dates"] = features["night_dates"].astype(int)
    features["office_dates"] = features["office_dates"].astype(int)

    user_night_total = features.groupby("user_id")["night_dwell_s"].transform("sum")
    user_office_total = features.groupby("user_id")["office_dwell_s"].transform("sum")

    features["night_dwell_share"] = np.where(
        user_night_total > 0,
        features["night_dwell_s"] / user_night_total,
        0.0,
    )
    features["office_dwell_share"] = np.where(
        user_office_total > 0,
        features["office_dwell_s"] / user_office_total,
        0.0,
    )

    features["night_dwell_h"] = features["night_dwell_s"] / 3600.0
    features["office_dwell_h"] = features["office_dwell_s"] / 3600.0
    return features


def top_candidates_for(
    features,
    *,
    label,
    min_dates,
):
    if label == "home":
        share_col = "night_dwell_share"
        dates_col = "night_dates"
        dwell_col = "night_dwell_s"
    elif label == "office":
        share_col = "office_dwell_share"
        dates_col = "office_dates"
        dwell_col = "office_dwell_s"
    else:
        raise ValueError(label)

    ranked = features[
        (features["stay_count"] >= 2)
        & (features[dates_col] >= min_dates)
        & (features[dwell_col] > 0)
    ].copy()

    if ranked.empty:
        return ranked

    ranked = ranked.sort_values(
        ["user_id", share_col, dates_col, dwell_col, "stay_count"],
        ascending=[True, False, False, False, False],
        kind="stable",
    )
    ranked[f"{label}_rank"] = ranked.groupby("user_id").cumcount() + 1
    return top_with_margin(
        ranked,
        label=label,
        share_col=share_col,
    )


def window_stability_row(
    top,
    *,
    baseline_top,
    label,
    variant,
    share_col,
    margin_col,
    dates_col,
):
    if top.empty:
        return {
            "variant": variant,
            "supported_users": 0,
            "shared_with_baseline": 0,
            "same_top_location_rate": np.nan,
            "median_share": np.nan,
            "median_margin": np.nan,
            "median_dates": np.nan,
        }

    current = top[["user_id", "location_id"]].rename(
        columns={"location_id": "current_location_id"}
    )
    base = baseline_top[["user_id", "location_id"]].rename(
        columns={"location_id": "baseline_location_id"}
    )
    shared = current.merge(base, on="user_id", how="inner")

    same_rate = (
        float(
            (shared["current_location_id"] == shared["baseline_location_id"]).mean()
        )
        if len(shared)
        else np.nan
    )

    return {
        "variant": variant,
        "supported_users": len(top),
        "shared_with_baseline": len(shared),
        "same_top_location_rate": same_rate,
        "median_share": float(top[share_col].median()),
        "median_margin": float(top[margin_col].median()),
        "median_dates": float(top[dates_col].median()),
    }


HOME_WINDOW_VARIANTS = [
    ("20-06", 20, 6),
    ("21-06", 21, 6),
    ("22-06", 22, 6),
]
OFFICE_WINDOW_VARIANTS = [
    ("08-17", 8, 17),
    ("09-17", 9, 17),
    ("09-18", 9, 18),
]

home_window_rows = []
for name, start_hour, end_hour in HOME_WINDOW_VARIANTS:
    features_variant = build_semantic_features_for_windows(
        home_start_hour=start_hour,
        home_end_hour=end_hour,
        office_start_hour=OFFICE_START_HOUR,
        office_end_hour=OFFICE_END_HOUR,
    )
    top_variant = top_candidates_for(
        features_variant,
        label="home",
        min_dates=2,
    )
    home_window_rows.append(
        window_stability_row(
            top_variant,
            baseline_top=home_top,
            label="home",
            variant=name,
            share_col="night_dwell_share",
            margin_col="home_share_margin",
            dates_col="night_dates",
        )
    )

office_window_rows = []
for name, start_hour, end_hour in OFFICE_WINDOW_VARIANTS:
    features_variant = build_semantic_features_for_windows(
        home_start_hour=HOME_NIGHT_START_HOUR,
        home_end_hour=HOME_NIGHT_END_HOUR,
        office_start_hour=start_hour,
        office_end_hour=end_hour,
    )
    top_variant = top_candidates_for(
        features_variant,
        label="office",
        min_dates=2,
    )
    office_window_rows.append(
        window_stability_row(
            top_variant,
            baseline_top=office_top,
            label="office",
            variant=name,
            share_col="office_dwell_share",
            margin_col="office_share_margin",
            dates_col="office_dates",
        )
    )

print("HOME window stability:")
display(pd.DataFrame(home_window_rows))

print("OFFICE window stability:")
display(pd.DataFrame(office_window_rows))


recurring_user_count = int(
    semantic_locations.loc[
        semantic_locations["stay_count"] >= 2,
        "user_id",
    ].nunique()
)

def emission_grid(
    features,
    *,
    label,
    min_dates_values,
    min_share_values,
    min_margin_values,
):
    if label == "home":
        share_col = "night_dwell_share"
        margin_col = "home_share_margin"
    elif label == "office":
        share_col = "office_dwell_share"
        margin_col = "office_share_margin"
    else:
        raise ValueError(label)

    rows = []
    for min_dates in min_dates_values:
        top = top_candidates_for(
            features,
            label=label,
            min_dates=min_dates,
        )

        for min_share in min_share_values:
            for min_margin in min_margin_values:
                emitted = top[
                    (top[share_col] >= min_share)
                    & (top[margin_col] >= min_margin)
                ]
                rows.append(
                    {
                        "min_dates": min_dates,
                        "min_share": min_share,
                        "min_margin": min_margin,
                        "emitted_users": len(emitted),
                        "cohort_coverage": len(emitted) / len(candidate_users),
                        "recurring_user_coverage": (len(emitted) / recurring_user_count if recurring_user_count else np.nan),
                    }
                )
    return pd.DataFrame(rows)


home_emission_sensitivity = emission_grid(
    semantic_features,
    label="home",
    min_dates_values=[2, 3, 5],
    min_share_values=[0.4, 0.5, 0.6],
    min_margin_values=[0.1, 0.2, 0.3],
)

office_emission_sensitivity = emission_grid(
    semantic_features,
    label="office",
    min_dates_values=[2, 3, 5],
    min_share_values=[0.2, 0.3, 0.4],
    min_margin_values=[0.05, 0.10, 0.20],
)

print("HOME emission sensitivity:")
display(home_emission_sensitivity)

print("OFFICE emission sensitivity:")
display(office_emission_sensitivity)

# -------------------------------------------------------------------
# Candidate v2 emissions under the old v1 scoring gates (fixed control)
# -------------------------------------------------------------------

def apply_candidate_gate(
    features,
    *,
    label,
    min_dates,
    min_share,
    min_margin,
):
    top = top_candidates_for(
        features,
        label=label,
        min_dates=min_dates,
    )

    if top.empty:
        return top

    if label == "home":
        share_col = "night_dwell_share"
        margin_col = "home_share_margin"
    elif label == "office":
        share_col = "office_dwell_share"
        margin_col = "office_share_margin"
    else:
        raise ValueError(label)

    emitted = top[
        (top[share_col] >= min_share)
        & (top[margin_col] >= min_margin)
    ].copy()

    emitted["label"] = label.upper()
    emitted["relevant_dwell_share"] = emitted[share_col]
    emitted["share_margin"] = emitted[margin_col]

    date_col = "night_dates" if label == "home" else "office_dates"
    emitted["relevant_dates"] = emitted[date_col].astype(int)

    support_factor = np.minimum(
        emitted["relevant_dates"].to_numpy(dtype=float) / 5.0,
        1.0,
    )

    emitted["evidence_strength"] = (
        emitted["relevant_dwell_share"].to_numpy(dtype=float)
        + emitted["share_margin"].to_numpy(dtype=float)
        + support_factor
    ) / 3.0

    return emitted


candidate_home_emitted = apply_candidate_gate(
    semantic_features,
    label="home",
    min_dates=3,
    min_share=0.50,
    min_margin=0.20,
)

candidate_office_emitted = apply_candidate_gate(
    semantic_features,
    label="office",
    min_dates=3,
    min_share=0.30,
    min_margin=0.10,
)

candidate_emission_counts = pd.Series(
    {
        "HOME": len(candidate_home_emitted),
        "OFFICE": len(candidate_office_emitted),
    },
    name="candidate_v2_emitted_users",
)

print("\nTimezone-aware emissions under the v1 scoring gates (audit control):")
display(candidate_emission_counts.to_frame())


### Cách đọc share, margin và support

`relevant_dwell_share`: location chiếm bao nhiêu HOME/OFFICE evidence.

`share_margin`: top location hơn runner-up bao nhiêu.

`relevant_dates`: evidence lặp qua bao nhiêu ngày.

```text
share cao + margin thấp → hai locations cạnh tranh
margin cao + ít dates → evidence còn mỏng
nhiều dates + share thấp → behavior phân tán
```

Final rule dùng gates trên raw evidence; `evidence_strength` chỉ là summary, không phải probability.


### 6.2 Scoring rule dùng làm audit control

Giữ v1 controls:

```text
HOME:   21–06 / >=3 dates / share>=0.50 / margin>=0.20
OFFICE: Mon–Fri 09–17 / >=3 dates / share>=0.30 / margin>=0.10
```

Sau Run All, nếu window stability vẫn cao và emission sensitivity thay đổi mượt khi strictness tăng, ta có lý do giữ rule.

`evidence_strength = (share + margin + support_factor) / 3`, với `support_factor=min(dates/5,1)`.

Score này không phải xác suất.


### 6.3 So sánh notebook mới với production CP2 v1

Production v1 vẫn dùng Beijing-radius contract; notebook cuối dùng per-stay timezone và không geography-filter user.

Vì vậy comparison không phải parity assertion mà là migration impact check.

Aggregate counts giống nhau vẫn chưa đủ; khi migrate phải compare exact user IDs, location IDs và evidence fields.


In [ ]:
# Current production v1 reference.
# This is intentionally NOT asserted equal to the timezone-v2 notebook candidate.

production_v1_config = HomeOfficeConfig()
production_v1_labels = infer_home_office(
    stays,
    config=production_v1_config,
)

production_v1_counts = (
    production_v1_labels["label"]
    .value_counts()
    .reindex(["HOME", "OFFICE"], fill_value=0)
)

migration_comparison = pd.DataFrame(
    {
        "production_v1": [
            int(production_v1_counts["HOME"]),
            int(production_v1_counts["OFFICE"]),
        ],
        "timezone_aware_notebook": [
            int(candidate_emission_counts["HOME"]),
            int(candidate_emission_counts["OFFICE"]),
        ],
    },
    index=["HOME", "OFFICE"],
)

migration_comparison["delta_candidate_minus_v1"] = (
    migration_comparison["timezone_aware_notebook"]
    - migration_comparison["production_v1"]
)

display(migration_comparison)

print("Production v1 rows:", len(production_v1_labels))
print(
    "Production v1 unique users:",
    production_v1_labels["user_id"].nunique(),
)
print(
    "Timezone-aware notebook unique emitted users:",
    len(
        set(candidate_home_emitted.get("user_id", pd.Series(dtype=str)))
        | set(candidate_office_emitted.get("user_id", pd.Series(dtype=str)))
    ),
)

print(
    "\nSTATUS: comparison only. "
    "A difference is expected to trigger review, not an assertion failure."
)


### 6.4 Production migration gate

Trình tự đúng:

```text
1. Run All notebook final
2. review timezone coverage
3. review clustering sensitivity
4. review HOME/OFFICE sensitivity
5. implement timezone semantics trong src/
6. RED / regression tests
7. full-release direct-model parity
8. HTTP parity
```

Cho tới khi hoàn tất, production v1 là historical reference.


In [ ]:
print("PRODUCTION MIGRATION STATUS: NOT YET APPLIED")
print("Reason: src/geolife/model/home_office.py still implements the older Beijing-radius contract.")
print("Next: Run All this final notebook, review measured outputs, then migrate src/ with tests and parity.")


### Tại sao không ép production parity ngay?

Parity chỉ có ý nghĩa khi notebook và production thực hiện cùng contract.

Hiện tại notebook dùng per-stay IANA timezone + all resolved stays; production v1 dùng Beijing-radius cohort.

Phải chốt output notebook mới trước, sau đó migrate production rồi mới biến parity thành hard gate.


In [ ]:
final_summary = pd.Series(
    {
        "cp1_stays": int(len(stays)),
        "users_with_cp1_stays": int(stays["user_id"].nunique()),
        "timezone_resolved_stays": int(len(stays_semantic)),
        "timezone_resolved_users": int(stays_semantic["user_id"].nunique()),
        "distinct_timezones": int(stays_semantic["timezone_id"].nunique()),
        "semantic_locations": int(len(semantic_locations)),
        "recurring_locations": int((semantic_locations["stay_count"] >= 2).sum()),
        "users_with_recurring_location": int(
            semantic_locations.loc[semantic_locations["stay_count"] >= 2, "user_id"].nunique()
        ),
        "home_emitted_users": int(len(candidate_home_emitted)),
        "office_emitted_users": int(len(candidate_office_emitted)),
        "unique_emitted_users": int(
            len(
                set(candidate_home_emitted.get("user_id", pd.Series(dtype=str)))
                | set(candidate_office_emitted.get("user_id", pd.Series(dtype=str)))
            )
        ),
    },
    name="measured_value",
)
display(final_summary.to_frame())
print("\nInterpretation: timezone resolution defines local clock; recurrence + behavioral gates decide emit vs abstain.")


## 7. Kết luận — cách đọc pipeline cuối

```text
5,821 frozen CP1 stays
   ↓
coordinate → IANA timezone
   ↓
UTC → local time của từng stay
   ↓
DBSCAN benchmark vs complete-link
   ↓
HOME 21–06 local
OFFICE weekday 09–17 local
   ↓
share + margin + repeated-date support
   ↓
emit hoặc abstain
```

Notebook không còn cần Beijing reference point để xác định timezone và không dùng `Asia/Shanghai` như geography scope.

Abstention đến từ evidence thật sự cần cho Home/Office: history quá ít, không có recurring location, evidence không repeat đủ ngày, share thấp hoặc margin nhỏ.

Cell `final_summary` cho measured counts sau khi Run All. Đây là **coverage**, không phải accuracy.

Production `src/` vẫn cần migration riêng trước khi timezone-aware behavior được gọi là production-frozen.
